# Layer-wise Class Separation R^2 (BCIC 2A)

`layer_wise_class_separation_BCIC2A.py`가 slicing_mode/feature_mode 조합별로 저장한 4개의 CSV를 불러와서,

- x축: layer
- y축: `class_separation_r2` (Kornblith et al., 2021, Eq. 11)
- 한 그래프 안에 6개 method(frozen, linear_probe, finetune, LORA, VERA, dynamic_pearl)를 겹쳐서 표시

형태로 그립니다. `layer_wise_feature_distance_plots.ipynb`와 동일한 구조이며, 아래 `SPLIT_TO_PLOT` 값만 `'train'` / `'valid'` / `'test'`로 바꾸면 해당 split의 그래프를 그릴 수 있습니다.

feature distance 그래프와 달리 이 metric은 frozen을 기준으로 다른 method와 '비교'하는 게 아니라 각 method 자신의 feature가 클래스를 얼마나 잘 분리하는지를 독립적으로 나타내므로, frozen도 다른 method와 동일하게 하나의 선으로 표시합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 0. 설정값
# ==========================================
LOG_DIR = "/workspace/EEGPT/dataset/BCIC-2A/layer_wise"
CSV_DATE_TAG = "260930"

# 예: 260930_layer_wise_class_separation_raw_flatten.csv
def csv_path(slicing_mode, feature_mode):
    return (
        f"{LOG_DIR}/{CSV_DATE_TAG}_layer_wise_class_separation_"
        f"{slicing_mode}_{feature_mode}.csv"
    )

# (slicing_mode, feature_mode) 4가지 조합
COMBOS = [
    ("raw", "flatten"),
    ("raw", "pooling"),
    ("sliced", "flatten"),
    ("sliced", "pooling"),
]

METRIC = "class_separation_r2"

# 지금은 test set만, 나중에 'train' 또는 'valid'로 바꿔서 재실행하면 됨
SPLIT_TO_PLOT = "test"

# CSV의 method 값 -> 범례에 표시할 라벨 매핑
# (feature distance와 달리 frozen도 비교 대상 method와 동등하게 하나의 선으로 표시)
METHOD_LABELS = {
    "frozen": "Frozen",
    "linear_probe": "Linear probe",
    "finetune": "Fine tuning",
    "LORA": "LORA",
    "VERA": "VERA",
    "dynamic_pearl": "Dynamic Pearl",
}

METHOD_STYLES = {
    "Frozen":        {"color": "#808080", "linestyle": (0, (1, 1)),         "marker": "x"},
    "Linear probe":  {"color": "#2A78D6", "linestyle": (0, (3, 1, 1, 1)), "marker": "v"},
    "VERA":          {"color": "#EDA100", "linestyle": "--",             "marker": "s"},
    "Pearl":         {"color": "#1BAF7A", "linestyle": "-.",             "marker": "^"},
    "Fine tuning":   {"color": "#4A3AA7", "linestyle": ":",              "marker": "D"},
    "Dynamic Pearl": {"color": "#E34948", "linestyle": "-",              "marker": "o"},
    "LORA":          {"color": "#008300", "linestyle": (0, (5, 1)),      "marker": "P"},
}

In [ ]:
# ==========================================
# 1. CSV 로드
# ==========================================
def load_class_separation_csv(slicing_mode, feature_mode):
    path = csv_path(slicing_mode, feature_mode)
    df = pd.read_csv(path)
    return df

In [ ]:
# ==========================================
# 2. 그래프 함수 (x축=layer, y축=class_separation_r2, method별 선 오버레이)
# ==========================================
def plot_class_separation(df, split, slicing_mode, feature_mode, ax=None):
    sub = df[df["split"] == split]

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    for method_key, label in METHOD_LABELS.items():
        m = sub[sub["method"] == method_key].sort_values("layer")
        if m.empty:
            continue
        style = METHOD_STYLES.get(label, {})
        ax.plot(m["layer"], m[METRIC], label=label, **style)

    ax.set_xlabel("Layer")
    ax.set_ylabel("Class separation $R^2$")
    ax.set_title(f"Class separation $R^2$\n({slicing_mode}+{feature_mode}, {split} set)")
    ax.grid(alpha=0.3)
    ax.legend()
    return ax

In [ ]:
# ==========================================
# 3. 4개 조합 전체 그래프 그리기
# ==========================================
for slicing_mode, feature_mode in COMBOS:
    df = load_class_separation_csv(slicing_mode, feature_mode)

    plot_class_separation(df, SPLIT_TO_PLOT, slicing_mode, feature_mode)
    plt.tight_layout()
    plt.show()